<a href="https://colab.research.google.com/github/tillurakesh3/Rakesh_INFO5731_Fall2024/blob/main/INFO5731_Assignment_3_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 3**

**This exercise will provide a valuable learning experience in working with text data and extracting features using various topic modeling algorithms. Key concepts such as Latent Dirichlet Allocation (LDA), Latent Semantic Analysis (LSA) and BERTopic.**



**Expectations**:

*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

**Total points**: 100


NOTE: The output should be presented well to get **full points**

**Late submissions will have a penalty of 10% of the marks for each day of late submission, and no requests will be answered. Manage your time accordingly.**


# **Question 1 (20 Points)**

**Dataset**: 20 Newsgroups dataset

**Dataset Link**: https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html

**Consider Random 2000 rows only**

Generate K=10 topics by using LDA and LSA,
then calculate the coherence score and determine the optimal K value based on that score. Further, summarize and visualize each topic in your own words.


In [8]:
!pip install bertopic datasets "openai>=1.0.0" gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 0.28.0
    Uninstalling openai-0.28.0:
      Successfully uninstalled openai-0.28.0


In [5]:
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from datasets import load_dataset

# 1. Load Data
# The original fetch_20newsgroups is encountering a 403 Forbidden error.
# Using the 'datasets' library as an alternative to load 20 Newsgroups data.
# The 'remove' parameter is handled implicitly by the vectorizer's stop_words and max_features.
# newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))

# Load the 20 Newsgroups dataset using the Hugging Face 'datasets' library
dataset = load_dataset("SetFit/20_newsgroups")

# Combine train and test splits to get a dataset equivalent to 'subset='all''
train_docs = list(dataset['train']['text'])
test_docs = list(dataset['test']['text'])
all_docs = train_docs + test_docs

# Sample 2000 rows as requested
docs = pd.Series(all_docs).sample(2000, random_state=42).tolist()

# 2. Vectorization
tf_vectorizer = CountVectorizer(stop_words='english', max_features=1000)
tf = tf_vectorizer.fit_transform(docs)

# 3. LDA Implementation (K=10)
lda_model = LatentDirichletAllocation(n_components=10, random_state=42)
lda_model.fit(tf)

# 4. LSA Implementation (K=10)
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
tfidf = tfidf_vectorizer.fit_transform(docs)
lsa_model = TruncatedSVD(n_components=10, random_state=42)
lsa_model.fit(tfidf)

# Helper to visualize topics
def print_topics(model, vectorizer, top_n=10):
    for idx, topic in enumerate(model.components_):
        print(f"Topic {idx}: " + ", ".join([vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-top_n - 1:-1]]))

print("LDA Topics:")
print_topics(lda_model, tf_vectorizer)

Repo card metadata block was not found. Setting CardData to empty.


LDA Topics:
Topic 0: file, output, key, program, entry, return, oname, stream, char, printf
Topic 1: god, think, people, say, just, don, know, question, believe, things
Topic 2: don, game, year, just, think, like, got, right, didn, time
Topic 3: know, thanks, car, just, new, drive, help, like, does, good
Topic 4: like, know, dod, use, just, don, anonymous, people, posting, edu
Topic 5: 10, 20, 16, 15, la, 12, 11, 00, pts, 30
Topic 6: people, don, like, just, make, think, israel, believe, better, fact
Topic 7: people, said, children, armenians, new, armenian, azerbaijan, hiv, university, started
Topic 8: windows, image, data, scsi, software, bit, use, version, program, pc
Topic 9: ax, max, pl, 3t, 1t, bhj, 7ey, giz, 2tm, qq


# **BERTopic**

The following question is designed to help you develop a feel for the way topic modeling works, the connection to the human meanings of documents.

Dataset from **Assignment 2** (text dataset).

> Dont use any custom datasets.


> Dataset must have 1000+ rows, no duplicates and null values



# **Question 2 (20 Points)**



Q2) **Generate K=10 topics by using BERTopic and then find the optimal K value using the coherence score. Interpret each topic and visualize the results appropriately.**

In [1]:
import pandas as pd
from bertopic import BERTopic
import nltk
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from nltk.tokenize import word_tokenize

# Download necessary tokenizer
nltk.download('punkt')

# 1. Load your specific dataset from Assignment 2
df = pd.read_csv('cleaned_research_data (1).csv')

# 2. Preprocessing (Requirement: 1000+ rows, no duplicates, no nulls)
df = df.dropna().drop_duplicates()

# Identify the text column (replace 'text_column_name' with your actual column name, e.g., 'text' or 'content')
# If you are unsure of the column name, df.columns[0] usually targets the first column.
text_column = df.columns[0]
docs = df[text_column].astype(str).tolist()

print(f"Total documents after cleaning: {len(docs)}")

# 3. Generate K=10 topics using BERTopic
topic_model = BERTopic(nr_topics=10, calculate_probabilities=True, verbose=True)
topics, probs = topic_model.fit_transform(docs)

# 4. Calculate Coherence Score to find optimal K
def get_coherence_score(model, documents, topics):
    # Tokenize documents for Gensim
    tokenized_docs = [word_tokenize(doc.lower()) for doc in documents]
    dictionary = Dictionary(tokenized_docs)

    # Extract the top words for each topic from BERTopic
    topic_words = []
    for topic in range(len(set(topics)) - (1 if -1 in topics else 0)):
        words = [word for word, prob in model.get_topic(topic)][:10]
        topic_words.append(words)

    # Calculate Coherence
    coherence_model = CoherenceModel(topics=topic_words,
                                     texts=tokenized_docs,
                                     dictionary=dictionary,
                                     coherence='c_v')
    return coherence_model.get_coherence()

# Display info and current coherence
current_coherence = get_coherence_score(topic_model, docs, topics)
print(f"Coherence Score for K=10: {current_coherence}")

# 5. Interpret and Visualize
# Get details of the 10 topics
print(topic_model.get_topic_info().head(11))

# Visualizations
fig_topics = topic_model.visualize_topics()
fig_barchart = topic_model.visualize_barchart(top_n_topics=10)

fig_topics.show()
fig_barchart.show()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
2026-04-14 03:26:10,890 - BERTopic - Embedding - Transforming documents to embeddings.


Total documents after cleaning: 4771


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/150 [00:00<?, ?it/s]

2026-04-14 03:27:11,662 - BERTopic - Embedding - Completed ✓
2026-04-14 03:27:11,663 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 03:27:33,476 - BERTopic - Dimensionality - Completed ✓
2026-04-14 03:27:33,477 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 03:27:34,637 - BERTopic - Cluster - Completed ✓
2026-04-14 03:27:34,638 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-14 03:27:34,745 - BERTopic - Representation - Completed ✓
2026-04-14 03:27:34,746 - BERTopic - Topic reduction - Reducing number of topics
2026-04-14 03:27:34,762 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-14 03:27:34,871 - BERTopic - Representation - Completed ✓
2026-04-14 03:27:34,875 - BERTopic - Topic reduction - Reduced number of topics from 98 to 10


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [ ]:
import openai
from bertopic.representation import OpenAI

# 1. Setup OpenAI representation
# IMPORTANT: Replace "YOUR_OPENAI_API_KEY" with your actual OpenAI API key.
# You can also load it securely from Colab secrets if preferred.
client = openai.OpenAI(api_key="YOUR_OPENAI_API_KEY")
representation_model = OpenAI(model="gpt-3.5-turbo", delay_in_seconds=2, chat=True)

# 2. Fit BERTopic with LLM representations
topic_model_llm = BERTopic(representation_model=representation_model)
topics, probs = topic_model_llm.fit_transform(docs)

# 3. Display new meaningful labels generated by GPT
# This will show the topic names as summarized by the GPT model.
display(topic_model_llm.get_topic_info())

In [4]:
!pip install example-library

ERROR: Could not find a version that satisfies the requirement example-library (from versions: none)
ERROR: No matching distribution found for example-library


# **Question 3 (25 points)**


**Dataset Link**: 20 Newsgroups Dataset (Random 2000 values)

Q3) Using the given dataset, modify the default representation model by integrating OpenAI's GPT model to generate meaningful summaries for each topic. Additionally, calculate the coherence score to determine the optimal number of topics and retrain the model accordingly.



Useful Link: https://maartengr.github.io/BERTopic/getting_started/representation/llm#truncating-documents

In [9]:
import openai
from bertopic.representation import OpenAI

# 1. Setup OpenAI representation
# IMPORTANT: Replace "YOUR_OPENAI_API_KEY" with your actual OpenAI API key.
# You can also load it securely from Colab secrets.
client = openai.OpenAI(api_key="YOUR_OPENAI_API_KEY")
representation_model = OpenAI(model="gpt-3.5-turbo", delay_in_seconds=2, chat=True)

# 2. Fit BERTopic with LLM representations
topic_model_llm = BERTopic(representation_model=representation_model)
topics, probs = topic_model_llm.fit_transform(docs)

# 3. Display new meaningful labels generated by GPT
# This will show the topic names as summarized by the GPT model.
display(topic_model_llm.get_topic_info())

AttributeError: module 'openai' has no attribute 'OpenAI'

# **Question 4 (35 Points)**


**BERTopic** allows for extensive customization, including the choice of embedding models, dimensionality reduction techniques, and clustering algorithms.

**Dataset Link**: 20 Newsgroups Dataset (Random 2000 values)

Q4)

Q4.1) **Modify the default BERTopic pipeline to use a different embedding model (e.g., Sentence-Transformers) and a different clustering algorithm (e.g., DBSCAN instead of HDBSCAN).

Q4.2) Compare the results of the custom embedding model with the default BERTopic model in terms of topic coherence and interpretability.

Q4.3) Visualize the topics and provide a qualitative analysis of the differences

**

Useful Link :https://www.pinecone.io/learn/bertopic/

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from umap import UMAP

# 4.1 Custom Pipeline
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')
cluster_model = DBSCAN(eps=0.5, min_samples=5) # Replacing HDBSCAN

custom_model = BERTopic(
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=cluster_model
)

topics, _ = custom_model.fit_transform(docs)

# 4.2 Qualitative Analysis Visualization
custom_model.visualize_hierarchy().show()

## Extra Question (5 Points)

**Compare the results generated by the four topic modeling algorithms (LDA, LSA, BERTopic, Modified BERTopic), which one is better? You should explain the reasons in details.**

**This question will compensate for any points deducted in this exercise. Maximum marks for the exercise is 100 points.**

In [ ]:
# Write your code here

# Mandatory Question

**Important: Reflective Feedback on this exercise**

Please provide your thoughts and feedback on the exercises you completed in this assignment.

Consider the following points in your response:

**Learning Experience:** Describe your overall learning experience in working with text data and extracting features using various topic modeling algorithms. Did you understand these algorithms and did the implementations helped in grasping the nuances of feature extraction from text data.

**Challenges Encountered:** Were there specific difficulties in completing this exercise?

Relevance to Your Field of Study: How does this exercise relate to the field of NLP?

**(Your submission will not be graded if this question is left unanswered)**



In [ ]:
# Your answer here (no code for this question, write down your answer as detail as possible for the above questions):

'''
Please write you answer here:

Learning Experience: This exercise highlighted the shift from "bag-of-words" models (LDA/LSA) to "contextual" models (BERTopic).
Implementing these helped me understand how $n$-dimensional embeddings capture meaning better than mere word frequency.

Challenges: Tuning DBSCAN was significantly harder than HDBSCAN because it requires a fixed eps value, which often resulted in too many documents being labeled as "noise" (Topic -1).

Relevance: In the field of NLP, topic modeling is the foundation for document organization, automated tagging, and understanding large-scale customer feedback without manual reading.




'''